# Compound Library Screening

**BioPipelines example** — SAR screening of a compound library against a protein–DNA complex using Boltz2. Predicted affinities are merged with compound metadata and plotted by substituent group.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[all]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba
!micromamba create -f Environments/biopipelines.yaml -y

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines.boltz2 import Boltz2

with Pipeline(project="Setup", job="InstallTools"):
    Boltz2.install()

## Cell 3: Compound Screening Pipeline

Screens a ChemDraw compound library against the Trp repressor–DNA complex.
Affinities are merged with substituent metadata and visualised with a scatter plot.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines.compound_library import CompoundLibrary
from biopipelines.boltz2 import Boltz2
from biopipelines.panda import Panda
from biopipelines.plot import Plot

with Pipeline(project="TrpRepressor", job="CompoundLibraryScreen"):
    TrpR = Sequence("MAQQSPYSAAMAEERHQEWLRFVDLLKNAYQNDLHLPLLNLMLTPDEREALGTRVRIVEELLRGEMSQRELKNELGAGIATITRGSNSLKAAPVELRQWLEEVLLKSD",
                    ids="TrpR")
    DNA = Sequence("TGTACTAGTTAACTAGTAC",
                   ids="dna")
    library = CompoundLibrary("./ExamplePipelines/compound_library_TrpR.cdxml")
    cofolded = Boltz2(proteins=Bundle(TrpR, TrpR),
                      dsDNA=DNA,
                      ligands=Each(library))
    merged = Panda(tables=[library.tables.compounds, cofolded.tables.affinity],
                   operations=[Panda.merge(),
                                Panda.calculate({"aff_uM": "10**affinity_pred_value"})])
    plots = Plot(Plot.Scatter(data=merged.tables.result,
                      x="R1",
                      y="aff_uM",
                      title="Predicted Affinity by R1 group",
                      xlabel="R1",
                      ylabel="Predicted Affinity [uM]"))
plots